# Долгосрочная память аптечного AI-ассистента · этап 1: детерминированное ядро

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/serge3emskov/pharma-ltm-assistant/blob/main/notebooks/01_core_demo.ipynb)

Ассистент аптеки общается с покупателем и помнит, что тот о себе рассказывал: какие препараты принимает, на что аллергия, есть ли хронические заболевания, беременность, для кого покупает. Архитектура повторяет подход из статьи [«Долгосрочная память для медицинского AI-ассистента»](https://habr.com/ru/articles/1090888/):

* **LLM понимает смысл.** Классифицирует запрос и извлекает из диалога структурированные события.
* **Код управляет состоянием.** Валидация, нормализация названий, ID, время, тип изменения, итоговый статус, выборка контекста.

```
write-path (фон):  диалог → [LLM: разделы → extractor] → Pydantic-валидация → нормализация → state machine → SQLite
read-path (синхр.): запрос → [LLM: тип запроса] → набор разделов → фильтр статусов и сроков → контекст с пометкой источника
```

В этом ноутбуке LLM-шаги **замоканы готовым JSON**, поэтому GPU не нужен. На этапе 2 их место займёт Qwen. Всё, что ниже, — код, который останется без изменений.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/serge3emskov/pharma-ltm-assistant.git"

if IN_COLAB:
    if not os.path.exists("pharma-ltm-assistant"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    ROOT = os.path.abspath("pharma-ltm-assistant")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{ROOT}/requirements.txt"], check=True)
else:
    ROOT = os.path.abspath("..")

SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
# После git pull Python не перечитывает уже импортированный пакет — выгружаем старую версию
for name in [m for m in sys.modules if m == "pharmaltm" or m.startswith("pharmaltm.")]:
    del sys.modules[name]
import pharmaltm

commit = subprocess.run(["git", "-C", ROOT, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print(f"pharmaltm {pharmaltm.__version__} · commit {commit or 'нет git'}")

pharmaltm 0.2.2 · commit 9fc6c49


In [2]:
import json
from datetime import datetime, timedelta, timezone

import pandas as pd
from IPython.display import Markdown, display

from pharmaltm import (ContextBuilder, MemoryStore, MemoryWriter, PurchaseRecord, QueryType,
                       Section, classify_query_rules)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)


class Clock:
    """Управляемые часы: чтобы показать, что происходит через недели и месяцы."""
    def __init__(self):
        self.now = datetime(2026, 9, 1, 10, 0, tzinfo=timezone.utc)
    def __call__(self):
        return self.now
    def advance(self, days):
        self.now += timedelta(days=days)


clock = Clock()
store = MemoryStore()                         # SQLite в памяти; для файла: MemoryStore("ltm.db")
writer = MemoryWriter(store, clock=clock)     # write-path
builder = ContextBuilder(store, clock=clock)  # read-path
USER = "buyer-001"


def dialog(text, extraction, dialog_id):
    """Реплика покупателя + ответ extractor-а (пока мок) → изменения памяти."""
    display(Markdown(f"**{clock.now:%d.%m.%Y}, покупатель:** {text}"))
    report = writer.write(USER, extraction, dialog_id)
    return pd.DataFrame(report.to_rows()).astype(object).fillna("")


def facts_table(subject=None):
    rows = [{"subject": f.subject, "section": f.section.value, "key": f.key, "status": f.status,
             "data": f.data, "raw_names": f.raw_names, "mentions": f.mentions,
             "last_confirmed": f.last_confirmed_at.date()} for f in store.list_facts(USER, subject)]
    return pd.DataFrame(rows)


def ask(text, purchases=None):
    """Запрос покупателя → тип запроса (пока правила) → контекст для основной модели."""
    q = classify_query_rules(text)
    bundle = builder.build(USER, q.query_types, q.subject, purchases=purchases)
    display(Markdown(f"**Запрос:** {text}  \n**Типы:** {[t.value for t in q.query_types]} · "
                     f"**субъект:** {q.subject} · **разделы:** {[s.value for s in bundle.sections]}"))
    print(bundle.render())
    return bundle

## 1. Write-path: из диалога в память

### Диалог 1. Первое упоминание и «подтверждено отсутствие»

Extractor вернул три факта и статус раздела. Название препарата он передаёт как есть («Энап»), а ключ факта «эналаприл» определяет код по справочнику.

In [3]:
dialog(
    "Врач назначил Энап от давления, пока не начала пить. Аллергий у меня нет. Мне 52.",
    {
        "facts": [
            {"section": "medications", "name": "Энап", "status": "prescribed", "indication": "давление",
             "evidence": "Врач назначил Энап от давления, пока не начала пить"},
            {"section": "chronic_conditions", "name": "давление", "evidence": "от давления"},
            {"section": "demographics", "attribute": "age", "value": "52", "evidence": "Мне 52"},
        ],
        "section_statuses": [
            {"section": "allergies", "status": "confirmed_absent", "evidence": "Аллергий у меня нет"},
        ],
    },
    "d1",
)

**01.09.2026, покупатель:** Врач назначил Энап от давления, пока не начала пить. Аллергий у меня нет. Мне 52.

,change,section,subject,key,before,after
0,first_mention,medications,self,эналаприл,,"{'status': 'prescribed', 'data': {'indication': 'давление'}}"
1,first_mention,chronic_conditions,self,артериальная гипертензия,,"{'status': 'active', 'data': {}}"
2,first_mention,demographics,self,age,,"{'status': 'active', 'data': {'value': 52}}"
3,section_confirmed_absent,allergies,self,,"{'state': None, 'needs_clarification': False}","{'state': 'confirmed_absent', 'needs_clarification': False}"


### Диалог 2. Изменение факта и отклонённое событие

* Дозировку «5 мг» из названия «Энап 5 мг» достаёт код, а не LLM.
* Второе событие extractor выдумал: раздела «семейный анамнез» в схеме нет. Событие отклонено валидацией, первое записано.

In [4]:
clock.advance(3)
dialog(
    "Начала пить Энап 5 мг, по утрам. У отца, кстати, был инфаркт.",
    {"facts": [
        {"section": "medications", "name": "Энап 5 мг", "status": "active", "frequency": "1 раз в день, утром",
         "evidence": "Начала пить Энап 5 мг, по утрам"},
        {"section": "family_history", "name": "инфаркт", "subject": "отец", "evidence": "У отца был инфаркт"},
    ]},
    "d2",
)

**04.09.2026, покупатель:** Начала пить Энап 5 мг, по утрам. У отца, кстати, был инфаркт.

,change,section,subject,key,before,after
0,update,medications,self,эналаприл,"{'status': 'prescribed', 'data': {'indication': 'давление'}}","{'status': 'active', 'data': {'indication': 'давление', 'frequency': '1 раз в день, ут..."
1,rejected,,,,"{'section': 'family_history', 'name': 'инфаркт', 'subject': 'отец', 'evidence': 'У отц...",<root>: Input tag 'family_history' found using 'section' does not match any of the exp...


### Диалог 3. Новая дозировка и аллергия, которая перекрывает прежнее «аллергий нет»

In [5]:
clock.advance(20)
dialog(
    "Врач увеличил Энап до 10 мг. И вспомнила: на пенициллин у меня была сыпь.",
    {"facts": [
        {"section": "medications", "name": "Энап", "dosage": "10 мг", "evidence": "Врач увеличил Энап до 10 мг"},
        {"section": "allergies", "name": "пенициллин", "reaction_type": "allergy", "reaction": "сыпь",
         "evidence": "на пенициллин у меня была сыпь"},
    ]},
    "d3",
)

**24.09.2026, покупатель:** Врач увеличил Энап до 10 мг. И вспомнила: на пенициллин у меня была сыпь.

,change,section,subject,key,before,after
0,update,medications,self,эналаприл,"{'status': 'active', 'data': {'dosage': '5 мг', 'frequency': '1 раз в день, утром', 'i...","{'status': 'active', 'data': {'dosage': '10 мг', 'frequency': '1 раз в день, утром', '..."
1,first_mention,allergies,self,пенициллины,,"{'status': 'active', 'data': {'reaction_type': 'allergy', 'reaction': 'сыпь'}}"
2,section_state_cleared,allergies,self,,"{'state': 'confirmed_absent', 'needs_clarification': False}","{'state': None, 'needs_clarification': False}"


### Диалог 4. Отмена препарата, новый препарат, симптом и покупка для ребёнка

In [6]:
clock.advance(10)
dialog(
    "Энап больше не пью, перевели на Лозап 50. Третий день болит голова. "
    "И дочке нужно от температуры: ей 7 лет, весит 24 кг, температура 38,5.",
    {"facts": [
        {"section": "medications", "name": "Энап", "polarity": "negative", "evidence": "Энап больше не пью"},
        {"section": "medications", "name": "Лозап 50 мг", "status": "active", "indication": "давление",
         "evidence": "перевели на Лозап 50"},
        {"section": "symptoms", "name": "болит голова", "duration": "3 дня", "evidence": "Третий день болит голова"},
        {"section": "demographics", "subject": "дочке", "attribute": "age", "value": 7, "evidence": "ей 7 лет"},
        {"section": "demographics", "subject": "дочке", "attribute": "weight_kg", "value": "24 кг",
         "evidence": "весит 24 кг"},
        {"section": "symptoms", "subject": "дочке", "name": "температура", "evidence": "температура 38,5"},
    ]},
    "d4",
)

**04.10.2026, покупатель:** Энап больше не пью, перевели на Лозап 50. Третий день болит голова. И дочке нужно от температуры: ей 7 лет, весит 24 кг, температура 38,5.

,change,section,subject,key,before,after
0,deactivation,medications,self,эналаприл,"{'status': 'active', 'data': {'dosage': '10 мг', 'frequency': '1 раз в день, утром', '...","{'status': 'stopped', 'data': {'dosage': '10 мг', 'frequency': '1 раз в день, утром', ..."
1,first_mention,medications,self,лозартан,,"{'status': 'active', 'data': {'indication': 'давление', 'dosage': '50 мг'}}"
2,first_mention,symptoms,self,головная боль,,"{'status': 'active', 'data': {'duration': '3 дня'}}"
3,first_mention,demographics,дочь,age,,"{'status': 'active', 'data': {'value': 7}}"
4,first_mention,demographics,дочь,weight_kg,,"{'status': 'active', 'data': {'value': 24.0}}"
5,first_mention,symptoms,дочь,повышенная температура,,"{'status': 'active', 'data': {}}"


### Текущее состояние и история

Текущее состояние отвечает на вопрос «что известно сейчас». История — «как это менялось». Эналаприл неактивен, но ни одна дозировка не потеряна.

In [7]:
facts_table()

,subject,section,key,status,data,raw_names,mentions,last_confirmed
0,self,allergies,пенициллины,active,"{'reaction': 'сыпь', 'reaction_type': 'allergy'}",[пенициллин],1,2026-09-24
1,self,chronic_conditions,артериальная гипертензия,active,{},[давление],1,2026-09-01
2,self,demographics,age,active,{'value': 52},[],1,2026-09-01
3,self,medications,эналаприл,stopped,"{'dosage': '10 мг', 'frequency': '1 раз в день, утром', 'indication': 'давление'}","[Энап, Энап 5 мг]",4,2026-10-04
4,self,medications,лозартан,active,"{'dosage': '50 мг', 'indication': 'давление'}",[Лозап 50 мг],1,2026-10-04
5,self,symptoms,головная боль,active,{'duration': '3 дня'},[болит голова],1,2026-10-04
6,дочь,demographics,age,active,{'value': 7},[],1,2026-10-04
7,дочь,demographics,weight_kg,active,{'value': 24.0},[],1,2026-10-04
8,дочь,symptoms,повышенная температура,active,{},[температура],1,2026-10-04


In [8]:
enap = store.get_fact(USER, "self", Section.MEDICATIONS, "эналаприл")
pd.DataFrame([{"date": e.created_at.date(), "change": e.change_type, "before": e.before, "after": e.after,
               "evidence": e.payload["evidence"]} for e in store.history(enap.fact_id)])

,date,change,before,after,evidence
0,2026-09-01,first_mention,None,"{'data': {'indication': 'давление'}, 'status': 'prescribed'}","Врач назначил Энап от давления, пока не начала пить"
1,2026-09-04,update,"{'data': {'indication': 'давление'}, 'status': 'prescribed'}","{'data': {'dosage': '5 мг', 'frequency': '1 раз в день, утром', 'indication': 'давлени...","Начала пить Энап 5 мг, по утрам"
2,2026-09-24,update,"{'data': {'dosage': '5 мг', 'frequency': '1 раз в день, утром', 'indication': 'давлени...","{'data': {'dosage': '10 мг', 'frequency': '1 раз в день, утром', 'indication': 'давлен...",Врач увеличил Энап до 10 мг
3,2026-10-04,deactivation,"{'data': {'dosage': '10 мг', 'frequency': '1 раз в день, утром', 'indication': 'давлен...","{'data': {'dosage': '10 мг', 'frequency': '1 раз в день, утром', 'indication': 'давлен...",Энап больше не пью


## 2. Read-path: какую часть памяти отдать модели

Тип запроса определяет набор разделов, правила статусов — какие факты из них нужны. Блок «уточнить» говорит основной модели, о чём спросить перед рекомендацией.

In [9]:
clock.advance(1)
_ = ask("Что можно выпить от головной боли?")

**Запрос:** Что можно выпить от головной боли?  
**Типы:** ['symptom_relief'] · **субъект:** self · **разделы:** ['symptoms', 'reported_diagnoses', 'medications', 'allergies', 'chronic_conditions', 'special_states', 'demographics', 'lifestyle']

### ПАМЯТЬ АССИСТЕНТА — со слов покупателя (документами не подтверждено)
Субъект: сам покупатель

#### Жалобы и симптомы
- головная боль [актуально] — длительность: 3 дня — со слов: «болит голова» — подтверждено 2026-10-04

#### Лекарства
- лозартан [принимает] — дозировка: 50 мг; показание: давление — со слов: «Лозап 50 мг» — подтверждено 2026-10-04

#### Аллергии и непереносимость
- пенициллины [актуально] — реакция: сыпь; тип: аллергия — со слов: «пенициллин» — подтверждено 2026-09-24

#### Хронические заболевания
- артериальная гипертензия [актуально] — со слов: «давление» — подтверждено 2026-09-01

#### Возраст, пол, вес
- возраст: 52 [на 2026-09-01]

### НУЖНО УТОЧНИТЬ ПЕРЕД РЕКОМЕНДАЦИЕЙ (этого нет в памяти; если покупатель уже сказал об этом в текущем диалоге — не переспрашивай)
- Беременность и лактация: нет данных


Эналаприла в контексте нет: он отменён, а вопрос про текущее лечение. Возраст 52, пол неизвестен, поэтому беременность попала в «уточнить». Тот же факт вернётся, если спросить про прошлую терапию:

In [10]:
_ = ask("Что я раньше принимала от давления?")

**Запрос:** Что я раньше принимала от давления?  
**Типы:** ['therapy_history'] · **субъект:** self · **разделы:** ['medications', 'reported_diagnoses', 'chronic_conditions']

### ПАМЯТЬ АССИСТЕНТА — со слов покупателя (документами не подтверждено)
Субъект: сам покупатель

#### Лекарства
- эналаприл [больше не принимает] — дозировка: 10 мг; режим: 1 раз в день, утром; показание: давление — со слов: «Энап», «Энап 5 мг» — подтверждено 2026-10-04
- лозартан [принимает] — дозировка: 50 мг; показание: давление — со слов: «Лозап 50 мг» — подтверждено 2026-10-04

#### Хронические заболевания
- артериальная гипертензия [актуально] — со слов: «давление» — подтверждено 2026-09-01


In [11]:
_ = ask("Есть аналог Лозапа подешевле?")   # только лекарства и аллергии — симптомы и возраст не нужны

**Запрос:** Есть аналог Лозапа подешевле?  
**Типы:** ['analog'] · **субъект:** self · **разделы:** ['medications', 'allergies']

### ПАМЯТЬ АССИСТЕНТА — со слов покупателя (документами не подтверждено)
Субъект: сам покупатель

#### Лекарства
- лозартан [принимает] — дозировка: 50 мг; показание: давление — со слов: «Лозап 50 мг» — подтверждено 2026-10-04

#### Аллергии и непереносимость
- пенициллины [актуально] — реакция: сыпь; тип: аллергия — со слов: «пенициллин» — подтверждено 2026-09-24


In [12]:
_ = ask("Сколько давать дочке Нурофен?")   # ребёнку 7 лет: вес обязателен, про беременность не спрашиваем

**Запрос:** Сколько давать дочке Нурофен?  
**Типы:** ['dosing'] · **субъект:** дочь · **разделы:** ['demographics', 'medications', 'chronic_conditions', 'special_states']

### ПАМЯТЬ АССИСТЕНТА — со слов покупателя (документами не подтверждено)
Субъект: дочь

#### Возраст, пол, вес
- возраст: 7 [на 2026-10-04]
- вес, кг: 24.0 [на 2026-10-04]


## 3. Забывание по типу факта

В статье механизма забывания нет. Здесь срок актуальности задан политикой раздела: жалоба живёт 14 дней, аллергия — бессрочно, беременность через 280 дней надо переспросить. Устаревший факт уходит из контекста, но остаётся в истории.

In [13]:
clock.advance(20)
b = builder.build(USER, [QueryType.SYMPTOM_RELIEF])
print("головная боль в контексте:", "головная боль" in b.keys())
print("пенициллины в контексте:  ", "пенициллины" in b.keys())
print("головная боль в профиле:  ", "головная боль" in builder.build(USER, [QueryType.PROFILE_REVIEW]).keys())

головная боль в контексте: False
пенициллины в контексте:   True
головная боль в профиле:   True


## 4. Противоречие: «аллергий нет», хотя в памяти есть аллергия

Код не удаляет аллергию из-за одной реплики. Раздел помечается для уточнения, а основная модель получает и факт, и пометку.

In [14]:
dialog("Аллергий у меня никаких нет.",
       {"section_statuses": [{"section": "allergies", "status": "confirmed_absent", "evidence": "Аллергий нет"}]},
       "d5")

**25.10.2026, покупатель:** Аллергий у меня никаких нет.

,change,section,subject,key,before,after
0,conflict,allergies,self,,"{'state': None, 'needs_clarification': False}","{'state': None, 'needs_clarification': True, 'active_keys': ['пенициллины']}"


In [15]:
_ = ask("Подберите что-нибудь от боли в горле")

**Запрос:** Подберите что-нибудь от боли в горле  
**Типы:** ['symptom_relief'] · **субъект:** self · **разделы:** ['symptoms', 'reported_diagnoses', 'medications', 'allergies', 'chronic_conditions', 'special_states', 'demographics', 'lifestyle']

### ПАМЯТЬ АССИСТЕНТА — со слов покупателя (документами не подтверждено)
Субъект: сам покупатель

#### Лекарства
- лозартан [принимает] — дозировка: 50 мг; показание: давление — со слов: «Лозап 50 мг» — подтверждено 2026-10-04

#### Аллергии и непереносимость
- пенициллины [актуально] — реакция: сыпь; тип: аллергия — со слов: «пенициллин» — подтверждено 2026-09-24

#### Хронические заболевания
- артериальная гипертензия [актуально] — со слов: «давление» — подтверждено 2026-09-01

#### Возраст, пол, вес
- возраст: 52 [на 2026-09-01]

### НУЖНО УТОЧНИТЬ ПЕРЕД РЕКОМЕНДАЦИЕЙ (этого нет в памяти; если покупатель уже сказал об этом в текущем диалоге — не переспрашивай)
- Аллергии и непереносимость: противоречие: покупатель сказал «нет», но в памяти есть актуальные записи
- Беременность и лактация: нет данных


## 5. Два источника с пометкой происхождения

Слова покупателя (LTM) и история покупок по карте лояльности идут отдельными блоками. Отдельного resolver-а нет: если источники расходятся, основная модель видит оба и уточняет.

In [16]:
purchases = [PurchaseRecord("2026-10-01", "Лозап 50 мг", 2),
             PurchaseRecord("2026-09-28", "Нурофен Экспресс 200 мг")]
_ = ask("Можно ли ТераФлю вместе с моими таблетками?", purchases=purchases)

**Запрос:** Можно ли ТераФлю вместе с моими таблетками?  
**Типы:** ['drug_compatibility'] · **субъект:** self · **разделы:** ['medications', 'allergies', 'chronic_conditions', 'special_states', 'lifestyle']

### ПАМЯТЬ АССИСТЕНТА — со слов покупателя (документами не подтверждено)
Субъект: сам покупатель

#### Лекарства
- лозартан [принимает] — дозировка: 50 мг; показание: давление — со слов: «Лозап 50 мг» — подтверждено 2026-10-04

#### Аллергии и непереносимость
- пенициллины [актуально] — реакция: сыпь; тип: аллергия — со слов: «пенициллин» — подтверждено 2026-09-24

#### Хронические заболевания
- артериальная гипертензия [актуально] — со слов: «давление» — подтверждено 2026-09-01

### ДОКУМЕНТИРОВАНО — история покупок по карте лояльности
- 2026-10-01: «Лозап 50 мг» × 2 → лозартан
- 2026-09-28: «Нурофен Экспресс 200 мг» × 1 → ибупрофен
Если покупки расходятся со словами покупателя, не выбирай одну версию сам — уточни.

### НУЖНО УТОЧНИТЬ ПЕРЕД РЕКОМЕНДАЦИЕЙ (этого нет в памяти; если покупатель уже сказал об этом в текущем диалоге — не переспрашивай)
- Аллергии и непереносимость: противоречие: покупатель сказал «нет», но в памяти есть актуальные записи


## 6. Тесты по слоям

Как в статье, проверяется не финальный ответ, а каждый слой отдельно: контракт extractor-а, нормализация, переходы состояния, выборка контекста, E2E.

In [17]:
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                     cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-600:] or res.stderr[-600:])

........................................................................ [ 32%]
........................................................................ [ 64%]
........................................................................ [ 96%]
.........                                                                [100%]
225 passed in 0.41s



## Что дальше · этап 2

1. Qwen3-4B-Instruct в 4-bit (как в RAG-проекте) на месте моков: классификатор разделов, extractor с правилами только выбранного раздела, классификатор запроса.
2. Синтетические диалоги и golden set: эталонные события для каждого диалога.
3. Метрики по слоям: точность и полнота разделов у classifier, по полям у extractor, доля отклонённых событий, корректность контекста у read-path. Baseline — правила из `query_rules.py`.
4. Справочник: выгрузка ГРЛС вместо seed-словаря и детерминированная проверка противопоказаний и взаимодействий.

> Демонстрационный проект. Данные синтетические, ответы ассистента не являются медицинской рекомендацией.